# Aura AI — Existing Aura Checkpoint Inference Runtime

Serves the existing Aura Korean chat-SFT checkpoint from the private Kaggle inference dataset. Uses the matching tokenizer and private gateway/tunnel secrets; no new base model is initialized.


In [ ]:
import os, sys, time, uuid, hashlib, subprocess, urllib.request
from pathlib import Path

import torch
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

MODEL_ROOT = Path("/kaggle/input/aura-korean-chat-sft-v2-inference")
RUNTIME_CONFIG = Path("/kaggle/input/aura-ai-private-runtime")
TUNNEL_TOKEN = (RUNTIME_CONFIG / "cloudflare_tunnel_token.txt").read_text().strip()
API_SECRET = (RUNTIME_CONFIG / "gateway_secret.txt").read_text().strip()
if not API_SECRET or not TUNNEL_TOKEN:
    raise RuntimeError("Aura private runtime configuration is incomplete.")

sys.path.insert(0, str(MODEL_ROOT))
from aura.config import MODEL, TOKENIZER_PATH
from aura.models.transformer import AuraModel
from aura.tokenizer import AuraTokenizer, EOS, BOS

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, int(os.environ.get("AURA_TORCH_THREADS", "4"))))
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

tokenizer = AuraTokenizer.load(TOKENIZER_PATH)
if tokenizer.vocab_size != MODEL.vocab_size:
    raise RuntimeError(f"Tokenizer/model vocab mismatch: {tokenizer.vocab_size} != {MODEL.vocab_size}")

checkpoint_path = MODEL_ROOT / "data" / "checkpoints" / "best_chat_sft.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
tokenizer_sha = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
if checkpoint.get("tokenizer_sha256") != tokenizer_sha:
    raise RuntimeError("Aura checkpoint/tokenizer SHA-256 mismatch")

model = AuraModel(MODEL)
model.load_state_dict(checkpoint["model"], strict=True)
model.to(DEVICE).eval()
MODEL_INFO = {
    "name": "Aura Korean Chat SFT",
    "stage": checkpoint.get("stage", "chat_sft"),
    "step": checkpoint.get("step"),
    "validation_loss": checkpoint.get("validation_loss"),
    "device": str(DEVICE),
    "parameters": sum(p.numel() for p in model.parameters()),
}
print("AURA_MODEL_READY", MODEL_INFO, flush=True)

SYSTEM_PROMPT = (
    "너는 Aura AI다. 자연스럽고 정확한 한국어로 답한다. "
    "질문에 직접 답하고 근거 없는 사실을 만들지 않는다. "
    "불확실한 정보는 불확실하다고 밝히며, 이전 대화의 맥락과 조건을 유지한다."
)

app = FastAPI(title="Aura Inference", version="3.0.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://aura.koharu.live"],
    allow_methods=["GET", "POST"],
    allow_headers=["Authorization", "Content-Type"],
)

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = "aura"
    messages: list[Message]
    max_tokens: int = Field(default=160, ge=1, le=192)
    temperature: float = Field(default=0.55, ge=0.1, le=1.5)
    top_k: int = Field(default=30, ge=1, le=80)
    stream: bool = False

def authorize(authorization: str | None):
    if authorization != "Bearer " + API_SECRET:
        raise HTTPException(status_code=401, detail="Unauthorized")

def prompt_ids(messages: list[Message]) -> list[int]:
    ids = [BOS]
    for message in messages:
        role = message.role if message.role in {"system", "user", "assistant"} else "user"
        ids.extend(tokenizer.encode(f"<|{role}|>\n{message.content[:12000]}\n", add_special=False))
    ids.extend(tokenizer.encode("<|assistant|>\n", add_special=False))
    return ids[-MODEL.max_seq_len:]

@app.get("/health")
def health():
    return {"ok": True, "model": MODEL_INFO, "host": "kaggle"}

@app.get("/v1/models")
def models(authorization: str | None = Header(default=None)):
    authorize(authorization)
    return {"object": "list", "data": [{"id": "aura", "object": "model", "owned_by": "KCS"}]}

@app.post("/v1/chat/completions")
def completions(req: ChatRequest, authorization: str | None = Header(default=None)):
    authorize(authorization)
    if not req.messages or req.messages[-1].role != "user":
        raise HTTPException(status_code=400, detail="The last message must be from the user.")
    messages = [Message(role="system", content=SYSTEM_PROMPT)] + req.messages[-12:]
    ids = prompt_ids(messages)
    input_ids = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    started = time.time()
    try:
        with torch.inference_mode():
            output = model.generate(
                input_ids,
                max_new_tokens=req.max_tokens,
                temperature=req.temperature,
                top_k=req.top_k,
            )
        generated = output[0, len(ids):].tolist()
        if EOS in generated:
            generated = generated[:generated.index(EOS)]
        content = tokenizer.decode(generated).strip()
        if not content:
            content = "답변을 생성하지 못했습니다. 질문을 조금 다르게 입력해 주세요."
    except torch.cuda.OutOfMemoryError:
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        raise HTTPException(status_code=503, detail="Inference memory limit exceeded.")
    except Exception as exc:
        raise HTTPException(status_code=500, detail="Inference failed: " + type(exc).__name__)
    return {
        "id": "chatcmpl-" + uuid.uuid4().hex,
        "object": "chat.completion",
        "created": int(time.time()),
        "model": "aura",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": len(ids), "completion_tokens": len(generated), "total_tokens": len(ids) + len(generated)},
        "latency_seconds": round(time.time() - started, 3),
    }

cloudflared = Path("/kaggle/working/cloudflared")
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    cloudflared,
)
cloudflared.chmod(0o755)
tunnel_log = open("/kaggle/working/cloudflared.log", "w")
tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "run", "--token", TUNNEL_TOKEN],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
time.sleep(8)
if tunnel.poll() is not None:
    raise RuntimeError("Cloudflare Tunnel exited; inspect cloudflared.log")
print("AURA_TUNNEL_CONNECTED inference.koharu.live", flush=True)
uvicorn = __import__("uvicorn")
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info"))
await server.serve()
